# Fine-Tuning GPT-4.1-mini for Named Entity Recognition (NER)
---

In [7]:
!pip install openai


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
import json
from openai import OpenAI

### Initialise

In [2]:

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


###  Upload Training File

In [3]:
# Make sure the JSONL file is formatted according to OpenAI's fine-tuning schema
size = 1500
train_file_path = f"output_data/finetuned_data/train.jsonl"
val_file_path = f"output_data/finetuned_data/val.jsonl"
test_file_path = f"output_data/finetuned_data/test.jsonl"
train_file = client.files.create(
    file=open(train_file_path, "rb"),
    purpose="fine-tune"
)
val_file = client.files.create(
    file=open(val_file_path, "rb"),
    purpose="fine-tune"
)
training_file_id = train_file.id
val_file_id = val_file.id
print("Training file uploaded.")
print("Training File ID:", training_file_id)
print("Validation File ID:", val_file_id)


Training file uploaded.
Training File ID: file-WNKDDs7gRMJCiMBhDZbenf
Validation File ID: file-VTvuVPYqVdtabdmQwXeFwu


### Create Fine-Tuning Job

In [4]:
fine_tune_response = client.fine_tuning.jobs.create(
    training_file=training_file_id,
     validation_file=val_file_id,
    model= "gpt-4.1-mini-2025-04-14" #gpt-3.5-turbo #gpt-4o-mini-2024-07-18 #gpt-4.1-mini-2025-04-14
)

job_id = fine_tune_response.id
print("Fine-tuning job started.")
print("Job ID:", job_id)


Fine-tuning job started.
Job ID: ftjob-wYtLsl1qJVhCgqtLXshO6ekw


### Retrieve Fine-Tuning Job Info

In [13]:
# job = client.fine_tuning.jobs.retrieve(job_id) #"ftjob-m80aY0hp4Z2d6TOLD9zrxVwd"
job = client.fine_tuning.jobs.retrieve(job_id) #"ftjob-m80aY0hp4Z2d6TOLD9zrxVwd"
print("📄 Fine-tuning job status:")
print(job)


📄 Fine-tuning job status:
FineTuningJob(id='ftjob-wYtLsl1qJVhCgqtLXshO6ekw', created_at=1766096031, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-mini-2025-04-14:digital-science-dimensions::CoGgLDPB', finished_at=1766096559, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-Naf8JqIqj78cDx1u4sa0mO5I', result_files=['file-MB1Byb7Yvyv25W9iL2ykKF'], seed=398815581, status='succeeded', trained_tokens=92502, training_file='file-WNKDDs7gRMJCiMBhDZbenf', validation_file='file-VTvuVPYqVdtabdmQwXeFwu', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)


In [14]:
job

FineTuningJob(id='ftjob-wYtLsl1qJVhCgqtLXshO6ekw', created_at=1766096031, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-mini-2025-04-14:digital-science-dimensions::CoGgLDPB', finished_at=1766096559, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-Naf8JqIqj78cDx1u4sa0mO5I', result_files=['file-MB1Byb7Yvyv25W9iL2ykKF'], seed=398815581, status='succeeded', trained_tokens=92502, training_file='file-WNKDDs7gRMJCiMBhDZbenf', validation_file='file-VTvuVPYqVdtabdmQwXeFwu', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)

### Generate Predictions from the Fine-Tuned Model

In [15]:
# (Note: Ensure the job is completed and `fine_tuned_model` is available before running this)

import time

# ⏳ Wait or poll until fine-tuning job is complete
while job.fine_tuned_model is None:
    print("⏳ Waiting for fine-tuned model to be ready...")
    time.sleep(10)
    job = client.fine_tuning.jobs.retrieve(job_id)

model_id = job.fine_tuned_model
print("✅ Fine-tuned model is ready:", model_id)


✅ Fine-tuned model is ready: ft:gpt-4.1-mini-2025-04-14:digital-science-dimensions::CoGgLDPB


In [16]:
client.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-wYtLsl1qJVhCgqtLXshO6ekw', created_at=1766096031, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-mini-2025-04-14:digital-science-dimensions::CoGgLDPB', finished_at=1766096559, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3), model='gpt-4.1-mini-2025-04-14', object='fine_tuning.job', organization_id='org-Naf8JqIqj78cDx1u4sa0mO5I', result_files=['file-MB1Byb7Yvyv25W9iL2ykKF'], seed=398815581, status='succeeded', trained_tokens=92502, training_file='file-WNKDDs7gRMJCiMBhDZbenf', validation_file='file-VTvuVPYqVdtabdmQwXeFwu', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=2.0, n_epochs=3))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None)

### Save Predictions

In [17]:
import json
def generate_predictions(input_file, output_file):
    count = 0

    with open(input_file, 'r', encoding='utf-8') as f:
        examples = [json.loads(line) for line in f]

    with open(output_file, 'w', encoding='utf-8') as out_f:
        for ex in examples:
            user_input = ex["messages"][1]["content"]
            system_prompt = ex["messages"][0]["content"]

            # Try to extract expected output, default to {}
            try:
                expected_output = json.loads(ex["messages"][2]["content"])
            except json.JSONDecodeError:
                expected_output = {}
                
            
            # Initialize predicted_output and raw_content
            predicted_output = None
            raw_content = None

            response = client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ],
                temperature=0
            )

            raw_content = response.choices[0].message.content
            try:
                predicted_output = json.loads(raw_content)
            except json.JSONDecodeError:
                predicted_output = None  # Keep as None, will store raw content instead

            result = {
                "input": user_input,
                "predicted": predicted_output,
                "raw_response": raw_content,
                "expected": expected_output,
            }

            out_f.write(json.dumps(result) + "\n")
            count += 1

    print(f"\n✅ {count} predictions saved to: {output_file}")



In [4]:

output_predictions_path = f"output_gpt4.1_ft.jsonl"

generate_predictions(test_file_path, output_predictions_path)

### Run Prediction Generation

### Evaluate predictions

In [7]:
import math

import json

def parse_value(v):
    try:
        return float(v)
    except:
        return None

def percent_error(expected, predicted):
    if expected is None or predicted is None or expected == 0:
        return None
    return abs((predicted - expected) / expected) * 100

def evaluate_mechanical_properties(row, tolerance_pct=20):
    expected = row.get("expected", {}).get("mechanical_properties", [{}])[0]
    predicted = row.get("predicted", {}).get("mechanical_properties", [{}])[0]

    results = {}

    for prop, exp_list in expected.items():
        pred_list = predicted.get(prop)

        if not isinstance(exp_list, list) or not isinstance(pred_list, list):
            continue
        
        exp_item = exp_list[0] if exp_list else None
        pred_item = pred_list[0] if pred_list else None

        if not exp_item or not pred_item or "value" not in exp_item:
            continue

        exp_val = parse_value(exp_item.get("value"))
        pred_val = parse_value(pred_item.get("value"))

        err = percent_error(exp_val, pred_val)
        match = (
            "Match" if err is not None and err <= tolerance_pct else
            "Mismatch" if err is not None else
            "NA"
        )

        results[prop] = match

    return results


def per_property_metrics(rows, tolerance_pct=20):
    metrics = {}  # {property: {TP, FP, FN}}

    for row in rows:
        results = evaluate_mechanical_properties(row, tolerance_pct)

        for prop, match in results.items():
            if prop not in metrics:
                metrics[prop] = {"TP": 0, "FP": 0, "FN": 0}

            if match == "Match":
                metrics[prop]["TP"] += 1
            elif match == "Mismatch":
                metrics[prop]["FP"] += 1
                metrics[prop]["FN"] += 1
            else:  # NA or no prediction
                metrics[prop]["FN"] += 1

    report = {}
    for prop, counts in metrics.items():
        TP, FP, FN = counts["TP"], counts["FP"], counts["FN"]
        precision = TP / (TP + FP) if (TP + FP) else 0
        recall = TP / (TP + FN) if (TP + FN) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

        report[prop] = {
            "precision": round(precision, 4),
            "recall": round(recall, 4),
            "f1": round(f1, 4),
            "TP": TP,
            "FP": FP,
            "FN": FN
        }
    return report


def evaluate_jsonl(file_path, tolerance_pct=10):
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line.strip()))

    report = per_property_metrics(rows, tolerance_pct)

    print("\n📊 Per-Property Scores (tolerance:", tolerance_pct, "%)")
    for prop, scores in report.items():
        print(f"\nProperty: {prop}")
        print(scores)

    return report

In [8]:

evaluate_jsonl(output_predictions_path)



📊 Per-Property Scores (tolerance: 10 %)

Property: tensile_strength_mpa
{'precision': 0.6667, 'recall': 0.6667, 'f1': 0.6667, 'TP': 4, 'FP': 2, 'FN': 2}

Property: yield_strength_mpa
{'precision': 0.5, 'recall': 0.5, 'f1': 0.5, 'TP': 3, 'FP': 3, 'FN': 3}

Property: elongation_pct
{'precision': 0.1667, 'recall': 0.1667, 'f1': 0.1667, 'TP': 1, 'FP': 5, 'FN': 5}


{'tensile_strength_mpa': {'precision': 0.6667,
  'recall': 0.6667,
  'f1': 0.6667,
  'TP': 4,
  'FP': 2,
  'FN': 2},
 'yield_strength_mpa': {'precision': 0.5,
  'recall': 0.5,
  'f1': 0.5,
  'TP': 3,
  'FP': 3,
  'FN': 3},
 'elongation_pct': {'precision': 0.1667,
  'recall': 0.1667,
  'f1': 0.1667,
  'TP': 1,
  'FP': 5,
  'FN': 5}}